# Preprocessing — MovieLens (`ml-latest-small`)

## Inputs

MovieLens `ml-latest-small` files on Google Drive at `/content/gdrive/MyDrive/ml-latest-small/`:
- `movies.csv` (~9,742 rows)
- `ratings.csv` (~100,836 rows)
- `tags.csv` (~3,683 rows)

## Outputs (on Google Drive)

- `recsys_artifacts/splits/train.csv`
- `recsys_artifacts/splits/val.csv`
- `recsys_artifacts/splits/test.csv`
- `recsys_artifacts/movies_with_tags.csv` (for content-based and hybrid notebooks)

Every downstream notebook reads these files. The base data files (`movies.csv`, `ratings.csv`, `tags.csv`) are touched only here.

## Setup (Colab)

In [1]:
from google.colab import drive
drive.mount('/content/gdrive')

Mounted at /content/gdrive


In [2]:
import os, random
import numpy as np

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
os.environ['PYTHONHASHSEED'] = str(SEED)

In [3]:
import pandas as pd
from pathlib import Path

In [4]:
DATA_DIR     = Path('/content/gdrive/MyDrive/ml-latest-small')
ARTIFACT_DIR = Path('/content/gdrive/MyDrive/recsys_artifacts')
SPLITS_DIR   = ARTIFACT_DIR / 'splits'
SPLITS_DIR.mkdir(parents=True, exist_ok=True)

print('Reading from:', DATA_DIR)
print('Writing to:  ', ARTIFACT_DIR)

Reading from: /content/gdrive/MyDrive/ml-latest-small
Writing to:   /content/gdrive/MyDrive/recsys_artifacts


## Load raw data

In [5]:
movies  = pd.read_csv(DATA_DIR / 'movies.csv')
ratings = pd.read_csv(DATA_DIR / 'ratings.csv')
tags    = pd.read_csv(DATA_DIR / 'tags.csv')

print(f'movies : {movies.shape}')
print(f'ratings: {ratings.shape}')
print(f'tags   : {tags.shape}')

movies : (9742, 3)
ratings: (100836, 4)
tags   : (3683, 4)


In [6]:
ratings.head()

,userId,movieId,rating,timestamp
0,1,1,4.0,964982703
1,1,3,4.0,964981247
2,1,6,4.0,964982224
3,1,47,5.0,964983815
4,1,50,5.0,964982931


## Filter sparse users


In [7]:
MIN_RATINGS_PER_USER = 5

rating_counts = ratings.groupby('userId').size()
keep_users = rating_counts[rating_counts >= MIN_RATINGS_PER_USER].index

print(f'Users before filter: {ratings["userId"].nunique()}')
print(f'Users with ≥ {MIN_RATINGS_PER_USER} ratings: {len(keep_users)}')

ratings = ratings[ratings['userId'].isin(keep_users)].copy()
print(f'Ratings after filter: {len(ratings):,}')

Users before filter: 610
Users with ≥ 5 ratings: 610
Ratings after filter: 100,836


## Leave-one-out split by timestamp

For each user:
- most recent rating → **test**
- second-most-recent → **validation**
- everything else → **train**

Ties in `timestamp` are broken by `movieId` so the assignment is deterministic across reruns.

In [8]:
ratings_sorted = ratings.sort_values(['userId', 'timestamp', 'movieId'])

# Position from the END within each user's history.
ratings_sorted['rev_pos'] = ratings_sorted.groupby('userId').cumcount(ascending=False)

test  = ratings_sorted[ratings_sorted['rev_pos'] == 0].drop(columns=['rev_pos']).reset_index(drop=True)
val   = ratings_sorted[ratings_sorted['rev_pos'] == 1].drop(columns=['rev_pos']).reset_index(drop=True)
train = ratings_sorted[ratings_sorted['rev_pos'] >= 2].drop(columns=['rev_pos']).reset_index(drop=True)

print(f'train: {len(train):>7,} ratings, {train["userId"].nunique()} users, {train["movieId"].nunique()} movies')
print(f'val  : {len(val):>7,} ratings, {val["userId"].nunique()} users, {val["movieId"].nunique()} movies')
print(f'test : {len(test):>7,} ratings, {test["userId"].nunique()} users, {test["movieId"].nunique()} movies')

train:  99,616 ratings, 610 users, 9681 movies
val  :     610 ratings, 610 users, 512 movies
test :     610 ratings, 610 users, 514 movies


In [9]:
# Sanity checks: per-user counts and temporal monotonicity.
assert (val.groupby('userId').size()  == 1).all(),  'val should be exactly 1 rating per user'
assert (test.groupby('userId').size() == 1).all(),  'test should be exactly 1 rating per user'

train_max = train.groupby('userId')['timestamp'].max().rename('train_max')
val_ts    = val.set_index('userId')['timestamp'].rename('val_ts')
test_ts   = test.set_index('userId')['timestamp'].rename('test_ts')
chk = pd.concat([train_max, val_ts, test_ts], axis=1).dropna()

assert (chk['val_ts']  >= chk['train_max']).all(), 'val timestamp must be ≥ last train timestamp'
assert (chk['test_ts'] >= chk['val_ts']).all(),    'test timestamp must be ≥ val timestamp'

print('OK: leave-one-out temporal ordering holds for every user.')

OK: leave-one-out temporal ordering holds for every user.


## Save splits to Google Drive

In [10]:
train.to_csv(SPLITS_DIR / 'train.csv', index=False)
val.to_csv(  SPLITS_DIR / 'val.csv',   index=False)
test.to_csv( SPLITS_DIR / 'test.csv',  index=False)

print(f'Saved train -> {SPLITS_DIR / "train.csv"}')
print(f'Saved val   -> {SPLITS_DIR / "val.csv"}')
print(f'Saved test  -> {SPLITS_DIR / "test.csv"}')

Saved train -> /content/gdrive/MyDrive/recsys_artifacts/splits/train.csv
Saved val   -> /content/gdrive/MyDrive/recsys_artifacts/splits/val.csv
Saved test  -> /content/gdrive/MyDrive/recsys_artifacts/splits/test.csv


## Build a movie metadata file for content-based and hybrid notebooks

Aggregates all user tags per movie into a single string, joined with the title and genres. The content-based notebook tokenizes this for TF-IDF or sentence embeddings; the hybrid GBRT notebook uses sentence embeddings of this text as a content feature.

In [11]:
# Aggregate all tags per movie into one space-separated string.
movie_tags = (tags.groupby('movieId')['tag']
                  .apply(lambda x: ' '.join(x.astype(str).str.lower()))
                  .reset_index()
                  .rename(columns={'tag': 'all_tags'}))

movies_with_tags = movies.merge(movie_tags, on='movieId', how='left')
movies_with_tags['all_tags'] = movies_with_tags['all_tags'].fillna('')
movies_with_tags['genres_text'] = movies_with_tags['genres'].str.replace('|', ' ', regex=False)
movies_with_tags['content_text'] = (
    movies_with_tags['title'].fillna('') + ' '
    + movies_with_tags['genres_text'].fillna('') + ' '
    + movies_with_tags['all_tags']
).str.strip()

out_path = ARTIFACT_DIR / 'movies_with_tags.csv'
movies_with_tags.to_csv(out_path, index=False)
print(f'Saved movies metadata -> {out_path}')
movies_with_tags.head(3)

Saved movies metadata -> /content/gdrive/MyDrive/recsys_artifacts/movies_with_tags.csv


,movieId,title,genres,all_tags,genres_text,content_text
0,1,Toy Story (1995),Adventure|Animation|Children|Comedy|Fantasy,pixar pixar fun,Adventure Animation Children Comedy Fantasy,Toy Story (1995) Adventure Animation Children ...
1,2,Jumanji (1995),Adventure|Children|Fantasy,fantasy magic board game robin williams game,Adventure Children Fantasy,Jumanji (1995) Adventure Children Fantasy fant...
2,3,Grumpier Old Men (1995),Comedy|Romance,moldy old,Comedy Romance,Grumpier Old Men (1995) Comedy Romance moldy old


## Split summary

Quick descriptive statistics. The downstream notebooks read `splits/train.csv`, `splits/val.csv`, `splits/test.csv` from Google Drive and should not re-run this preprocessing logic themselves.

In [12]:
print('=== Split summary ===')
print(f'Users in all 3 splits: {len(set(train["userId"]) & set(val["userId"]) & set(test["userId"])):,}')
print(f'Train rating mean: {train["rating"].mean():.3f}  std: {train["rating"].std():.3f}')
print(f'Val   rating mean: {val["rating"].mean():.3f}  std: {val["rating"].std():.3f}')
print(f'Test  rating mean: {test["rating"].mean():.3f}  std: {test["rating"].std():.3f}')
print()
print('Train ratings per user:')
print(train.groupby('userId').size().describe().to_string())

=== Split summary ===
Users in all 3 splits: 610
Train rating mean: 3.499  std: 1.042
Val   rating mean: 3.676  std: 1.062
Test  rating mean: 3.679  std: 1.103

Train ratings per user:
count     610.000000
mean      163.304918
std       269.480584
min        18.000000
25%        33.000000
50%        68.500000
75%       166.000000
max      2696.000000
